# Human in the Loop

In [1]:
import asyncio
from autogen_agentchat.agents import AssistantAgent

from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')
model_client = OpenAIChatCompletionClient(model='gpt-4o', api_key=api_key)

In [2]:
input() # We face this issue because in VScode python input function does not work as expected. and requires some extension.

 Hi


'Hi'

In [4]:
import asyncio
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent

from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination
from dotenv import load_dotenv
from autogen_agentchat.ui import Console
import os

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')
model_client = OpenAIChatCompletionClient(model='gpt-4o', api_key=api_key)


# Three agents for Storytelling
narrator = AssistantAgent(
    name='Narrator',
    model_client=model_client)
hero = AssistantAgent(
    name='Hero',
    model_client=model_client)
guide = AssistantAgent(
    name='Guide',
    model_client=model_client)

# Team with max_turns of 1

team = RoundRobinGroupChat(
    participants=[narrator, hero, guide],
    max_turns=1)


async def main():
    task = ' Write a 3 part story about a mysterious forest less than 30 words.'
    
    while True:
        stream = team.run_stream(task=task)
        await Console(stream)

        # Here is when we take the feedback from the user
        feedback = input('Please provide your feedback(type "exit" to stop): ')

        if feedback.lower().strip() == 'exit':
            break

        task = feedback # Next task is the feedback


In [5]:
await main()

---------- TextMessage (user) ----------
 Write a 3 part story about a mysterious forest less than 30 words.
---------- TextMessage (Narrator) ----------
Part 1: Shadows danced as whispers beckoned in the ancient forest.
Part 2: A hidden path revealed secrets lost in time.
Part 3: Courage found the heart of mystery, light dispelled shadows. 

TERMINATE


Please provide your feedback(type "exit" to stop):  IntrOduce cat in the stOry line


---------- TextMessage (user) ----------
IntrOduce cat in the stOry line
---------- TextMessage (Hero) ----------
Part 1: Shadows danced as whispers beckoned in the ancient forest, a curious cat followed.
Part 2: The cat unveiled a hidden path, revealing secrets lost in time.
Part 3: With the cat's guidance, courage found the heart of mystery, light dispelled shadows.

TERMINATE


Please provide your feedback(type "exit" to stop):  exit


In [6]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.base import Handoff
from autogen_agentchat.conditions import HandoffTermination, TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv
import asyncio
import os

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')
model_client = OpenAIChatCompletionClient(model='gpt-4o')


lazy_agent = AssistantAgent(
    name='Lazy_Agent',
    model_client=model_client,
    handoffs=[Handoff(target='user',message='Transfering to user')],
    system_message='if you cannot complete the task, just transfer it to user. When done, say "TERMINATE"'
)


handoff_termination = HandoffTermination('user')
text_termination = TextMentionTermination('TERMINATE')

termination_condition = handoff_termination | text_termination

lazy_agent_team = RoundRobinGroupChat(
    participants=[lazy_agent],
    termination_condition=termination_condition)




async def main():
    task = 'Give me the current weather of New York.'
    await Console(lazy_agent_team.run_stream(task=task),output_stats=True)
    feedback = "The weather is sunny" #input()
    # Here is when we take the feedback from the user
    await Console(lazy_agent_team.run_stream(task=feedback))

In [7]:
await main()

---------- TextMessage (user) ----------
Give me the current weather of New York.
---------- ToolCallRequestEvent (Lazy_Agent) ----------
[FunctionCall(id='call_FeeH4TqCcTUiBz8FhmwHdQmC', arguments='{}', name='transfer_to_user')]
[Prompt tokens: 69, Completion tokens: 11]
---------- ToolCallExecutionEvent (Lazy_Agent) ----------
[FunctionExecutionResult(content='Transfering to user', name='transfer_to_user', call_id='call_FeeH4TqCcTUiBz8FhmwHdQmC', is_error=False)]
---------- HandoffMessage (Lazy_Agent) ----------
Transfering to user
---------- Summary ----------
Number of messages: 4
Finish reason: Handoff to user from Lazy_Agent detected.
Total prompt tokens: 69
Total completion tokens: 11
Duration: 0.97 seconds
---------- TextMessage (user) ----------
The weather is sunny
---------- TextMessage (Lazy_Agent) ----------
Thank you for the update! If there's anything else you need, feel free to ask. TERMINATE
